#E-Commerce Customer and Products Modeling

Dim_Customer - RFM Analysis

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("E-Commerce Customer and Products Modeling")
    .enableHiveSupport()
    .getOrCreate()
)

2026-09-03 18:17:57,087 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
2026-09-03 18:17:57,115 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-03 18:18:00,959 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
customers = spark.read.parquet(
    "/ecom_modeling/cleaned/customers_clean"
)

orders = spark.read.parquet(
    "/ecom_modeling/cleaned/orders_clean"
)

order_items = spark.read.parquet(
    "/ecom_modeling/cleaned/order_items_clean"
)

order_payments = spark.read.parquet(
    "/ecom_modeling/cleaned/order_payments_clean"
)

In [16]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
order_payments.createOrReplaceTempView("order_payments")

#cleaning order_payments

In [39]:
duplicates = spark.sql("""
    SELECT
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value,
        COUNT(*) AS duplicate_count
    FROM order_payments
    GROUP BY
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value
    HAVING COUNT(*) > 1
""")

duplicates.show(20, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+---------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|duplicate_count|
+--------------------------------+------------------+------------+--------------------+-------------+---------------+
|2e2c60b99754ae1e4d8b18846cfec9f2|1                 |credit_card |4                   |542.66       |2              |
|881206af1a6a9cd34119b99782759dd4|1                 |boleto      |1                   |44.77        |2              |
|04f1827088d972a62224f5203a071500|1                 |credit_card |10                  |178.06       |2              |
|6ed3f553337457d7804e8c39b580ce89|1                 |credit_card |10                  |113.62       |2              |
|714b93bc4ed81c32fbf579f30619f3eb|2                 |voucher     |1                   |173.38       |2              |
|d513f9d572bf6a3a3c4e59252fe4f19c|1                 |cre

In [48]:
order_payments_cleaned = spark.sql("""
    SELECT DISTINCT
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value
    FROM order_payments
""")

order_payments_cleaned.createOrReplaceTempView(
    "order_payments_cleaned"
)

In [50]:
duplicates = spark.sql("""
    SELECT
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value,
        COUNT(*) AS duplicate_count
    FROM order_payments_cleaned
    GROUP BY
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value
    HAVING COUNT(*) > 1
""")

duplicates.show(20, truncate=False)

+--------+------------------+------------+--------------------+-------------+---------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|duplicate_count|
+--------+------------------+------------+--------------------+-------------+---------------+
+--------+------------------+------------+--------------------+-------------+---------------+



#Filter Valid Orders & Identify Customers

In [23]:
valid_customer_orders = spark.sql("""
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp
    FROM customers c
    INNER JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.order_status <> 'canceled'
""")

valid_customer_orders.show(5)

+--------------------+--------------------+------------------------+
|  customer_unique_id|            order_id|order_purchase_timestamp|
+--------------------+--------------------+------------------------+
|6457be0b331148fb5...|000aed2e25dbad2f9...|     2018-05-11 20:33:38|
|4125ca92d09b7d7a7...|008dd5e80ebf8f849...|     2018-03-20 20:31:23|
|5883c17380bc97092...|00cf47526e0f7920b...|     2017-02-09 13:21:46|
|bc63c4173e1894a96...|0114835d7f0b3f674...|     2017-08-31 16:17:50|
|cd6b577df45c00daa...|019aaee09698daf81...|     2017-08-10 14:04:58|
+--------------------+--------------------+------------------------+
only showing top 5 rows



In [25]:
valid_customer_orders.createOrReplaceTempView("valid_customer_orders")

In [73]:
customer_payments = spark.sql("""
    SELECT
        v.customer_unique_id,
        v.order_id,
        v.order_purchase_timestamp,
        SUM(p.payment_value) AS order_total_spend
    FROM valid_customer_orders v
    INNER JOIN order_payments_cleaned p
        ON v.order_id = p.order_id
    GROUP BY
        v.customer_unique_id,
        v.order_id,
        v.order_purchase_timestamp
""")

customer_payments.show(10)

+--------------------+--------------------+------------------------+-----------------+
|  customer_unique_id|            order_id|order_purchase_timestamp|order_total_spend|
+--------------------+--------------------+------------------------+-----------------+
|99c8fe6c03b909611...|245ac19080f071a4e...|     2017-12-09 00:10:59|            38.93|
|693b3203dcd4cef52...|4c77961d5a6909826...|     2017-09-22 22:33:22|            31.75|
|cb4253d36367737b6...|5784dd3f2dfc12119...|     2017-06-14 00:58:04|            83.14|
|77ea34bdf8e32a91d...|6defcba2f5df4f21c...|     2017-05-24 22:42:55|           101.12|
|28045a6a1ab1aab92...|74eb09c2340592100...|     2017-03-28 17:00:27|            54.34|
|73b186766ffa66af1...|8a1971d2df1c5e610...|     2018-01-02 13:11:53|            37.09|
|ed776320ce6ea9a8e...|9bf24c99c60f8ca32...|     2018-01-06 11:57:52|            55.10|
|518b5058581d593d2...|e1f58cebd92cdac4b...|     2018-08-23 14:59:37|            68.05|
|7ef15bb849ef9cfcc...|1a7af25ac2d2bf35b...|

In [74]:
customer_payments.createOrReplaceTempView("customer_payments")

In [ ]:
#Calculate RFM Metrics

In [75]:


rfm = spark.sql("""
    SELECT
        v.customer_unique_id,

        DATEDIFF(
            TO_DATE((SELECT MAX(order_purchase_timestamp)
                     FROM valid_customer_orders)),
            TO_DATE(MAX(v.order_purchase_timestamp))
        ) AS recency,

        COUNT(DISTINCT v.order_id) AS frequency,

        SUM(p.order_total_spend) AS monetary

    FROM valid_customer_orders v

    INNER JOIN customer_payments p
        ON v.order_id = p.order_id

    GROUP BY v.customer_unique_id

    HAVING SUM(p.order_total_spend) > 0
""")

rfm.show(10)

+--------------------+-------+---------+--------+
|  customer_unique_id|recency|frequency|monetary|
+--------------------+-------+---------+--------+
|dddcc50cec9aabfe4...|    383|        1|  176.78|
|8d0e0879191d761df...|    230|        1|  278.38|
|8a9f0cb78da35bb89...|    458|        1|  257.04|
|20688d09e8fcae954...|    350|        1|  119.84|
|63590d97e0d4877dc...|    279|        1|   57.10|
|e9610db1ce903b47e...|      9|        1|   95.72|
|a4f2995402f4f217f...|     48|        1|   68.40|
|55979bb000a56ddbe...|    241|        1|  147.15|
|f7d6e0dd60c793a66...|    292|        1|  123.39|
|d504041b9392f58cf...|     49|        1|  147.14|
+--------------------+-------+---------+--------+
only showing top 10 rows



In [77]:
rfm.createOrReplaceTempView("rfm")

In [88]:
spark.sparkContext.setLogLevel("ERROR")

In [89]:

rfm_scores = spark.sql("""
    SELECT
        customer_unique_id,
        recency,
        frequency,
        monetary,

        6 - NTILE(5) OVER (
            ORDER BY recency ASC
        ) AS recency_score,

        NTILE(5) OVER (
            ORDER BY frequency ASC
        ) AS frequency_score,

        NTILE(5) OVER (
            ORDER BY monetary ASC
        ) AS monetary_score

    FROM rfm
""")

rfm_scores.show(10)

+--------------------+-------+---------+--------+-------------+---------------+--------------+
|  customer_unique_id|recency|frequency|monetary|recency_score|frequency_score|monetary_score|
+--------------------+-------+---------+--------+-------------+---------------+--------------+
|317cfc692e3f86c45...|      9|        1|    9.59|            5|              1|             1|
|bd06ce0e06ad77a7f...|    355|        1|   10.07|            2|              4|             1|
|b33336f46234b24a6...|     74|        1|   10.89|            5|              1|             1|
|2878e5b88167faab1...|    309|        1|   11.63|            2|              4|             1|
|6f5b9d1cdccc4d28f...|    366|        1|   11.63|            2|              4|             1|
|809ca96e9696b9be5...|    464|        1|   12.28|            1|              5|             1|
|7859a40482024a3d0...|     95|        1|   12.39|            5|              1|             1|
|a2cab25fa1a8a53ba...|    103|        1|   12.89| 

In [79]:
rfm_scores.createOrReplaceTempView("rfm_scores")

In [3]:
spark.sparkContext.setLogLevel("ERROR")

In [87]:

final_rfm = spark.sql("""
    SELECT
        customer_unique_id,
        recency,
        frequency,
        monetary,
        recency_score,
        frequency_score,
        monetary_score,

        CONCAT(
            recency_score,
            frequency_score,
            monetary_score
        ) AS RFM_SCORE

    FROM rfm_scores
""")

final_rfm.show(20, truncate=False)

+--------------------------------+-------+---------+--------+-------------+---------------+--------------+---------+
|customer_unique_id              |recency|frequency|monetary|recency_score|frequency_score|monetary_score|RFM_SCORE|
+--------------------------------+-------+---------+--------+-------------+---------------+--------------+---------+
|317cfc692e3f86c45c95697c61c853a6|9      |1        |9.59    |5            |1              |1             |511      |
|bd06ce0e06ad77a7f681f1a4960a3cc6|355    |1        |10.07   |2            |4              |1             |241      |
|b33336f46234b24a613ad9064d13106d|74     |1        |10.89   |5            |1              |1             |511      |
|2878e5b88167faab17d4fb83a986d38b|309    |1        |11.63   |2            |4              |1             |241      |
|6f5b9d1cdccc4d28f0483a612edecacf|366    |1        |11.63   |2            |4              |1             |241      |
|809ca96e9696b9be5f69cd7ae803049d|464    |1        |12.28   |1  

In [81]:
final_rfm.createOrReplaceTempView("final_rfm")

In [100]:

customer_segments = spark.sql("""
    SELECT
        customer_unique_id,

        recency,
        frequency,
        monetary,

        recency_score,
        frequency_score,
        monetary_score,

        RFM_SCORE,

        CASE
            WHEN recency_score = 5
                 AND frequency_score BETWEEN 4 AND 5
                THEN 'Champions'

            WHEN recency_score BETWEEN 3 AND 4
                 AND frequency_score BETWEEN 4 AND 5
                THEN 'Loyal Customers'

            WHEN recency_score BETWEEN 4 AND 5
                 AND frequency_score BETWEEN 2 AND 3
                THEN 'Potential Loyalists'

            WHEN recency_score BETWEEN 1 AND 2
                 AND frequency_score BETWEEN 3 AND 4
                THEN 'At Risk'

            WHEN recency_score = 5
                 AND frequency_score = 1
                THEN 'New Customers'

            WHEN recency_score BETWEEN 1 AND 2
                 AND frequency_score BETWEEN 1 AND 2
                THEN 'Hibernating'

            ELSE 'Others'
        END AS Segment

    FROM final_rfm
""")

customer_segments.select(
    "customer_unique_id",
    "Segment",
    "recency_score",
    "frequency_score",
    "monetary_score",
     "RFM_SCORE"
).show(20, truncate=False)

+--------------------------------+-------------------+-------------+---------------+--------------+---------+
|customer_unique_id              |Segment            |recency_score|frequency_score|monetary_score|RFM_SCORE|
+--------------------------------+-------------------+-------------+---------------+--------------+---------+
|317cfc692e3f86c45c95697c61c853a6|New Customers      |5            |1              |1             |511      |
|bd06ce0e06ad77a7f681f1a4960a3cc6|At Risk            |2            |4              |1             |241      |
|b33336f46234b24a613ad9064d13106d|New Customers      |5            |1              |1             |511      |
|2878e5b88167faab17d4fb83a986d38b|At Risk            |2            |4              |1             |241      |
|6f5b9d1cdccc4d28f0483a612edecacf|At Risk            |2            |4              |1             |241      |
|809ca96e9696b9be5f69cd7ae803049d|Others             |1            |5              |1             |151      |
|7859a4048

#Customer_DIM

In [90]:
customer_details = spark.sql("""
    SELECT
        customer_unique_id,
        customer_city,
        customer_state
    FROM (
        SELECT
            customer_unique_id,
            customer_city,
            customer_state,
            ROW_NUMBER() OVER (
                PARTITION BY customer_unique_id
                ORDER BY customer_id
            ) AS rn
        FROM customers
    )
    WHERE rn = 1
""")

customer_details.show(10, truncate=False)

+--------------------------------+--------------+--------------+
|customer_unique_id              |customer_city |customer_state|
+--------------------------------+--------------+--------------+
|000c8bdb58a29e7115cfc257230fb21b|belo horizonte|MG            |
|0078bb0f0d23e922d08437b7d0e13907|belo horizonte|MG            |
|008ca52811784a181c8c88e8d66b49db|natal         |RN            |
|0098f4847541cc7d676f9a5efc64c0f0|belo horizonte|MG            |
|02bab436d042d111ec4b7c67cfab8835|ribeirao pires|SP            |
|02c82b1b79ccf9f38fef2fa51cbc8791|porto alegre  |RS            |
|03f8cc5eb864ffa50cb73bdf07a7b1e9|sao paulo     |SP            |
|04ebe7670da121a91344d0fd5547222f|fortaleza     |CE            |
|055ab020f1b15db97acb142cb9d893ae|brasilia      |DF            |
|05f5af0b8213fa1b6e28e7ed7a47a300|brasilia      |DF            |
+--------------------------------+--------------+--------------+
only showing top 10 rows



In [96]:
customer_details.createOrReplaceTempView("customer_details")
customer_segments.createOrReplaceTempView("customer_segments")

In [98]:
dim_customer = spark.sql("""
    SELECT
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,

        r.recency,
        r.frequency,
        r.monetary,

        r.recency_score,
        r.frequency_score,
        r.monetary_score,

        r.RFM_SCORE,
        r.Segment

    FROM customer_details c
    INNER JOIN customer_segments r
        ON c.customer_unique_id = r.customer_unique_id
""")

dim_customer.select(
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "RFM_SCORE",
    "Segment"
).show(20, truncate=False)

+--------------------------------+--------------------+--------------+---------+-------------------+
|customer_unique_id              |customer_city       |customer_state|RFM_SCORE|Segment            |
+--------------------------------+--------------------+--------------+---------+-------------------+
|317cfc692e3f86c45c95697c61c853a6|paulinia            |SP            |511      |New Customers      |
|bd06ce0e06ad77a7f681f1a4960a3cc6|sao paulo           |SP            |241      |At Risk            |
|b33336f46234b24a613ad9064d13106d|sao paulo           |SP            |511      |New Customers      |
|2878e5b88167faab17d4fb83a986d38b|sao paulo           |SP            |241      |At Risk            |
|6f5b9d1cdccc4d28f0483a612edecacf|sao paulo           |SP            |241      |At Risk            |
|809ca96e9696b9be5f69cd7ae803049d|santa isabel        |SP            |151      |Others             |
|7859a40482024a3d00041c4ca1434298|sao paulo           |SP            |511      |New Custome

In [99]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dim_customer = dim_customer.withColumn(
    "cust_key",
    row_number().over(
        Window.orderBy("customer_unique_id")
    )
)

dim_customer = dim_customer.select(
    "cust_key",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "recency",
    "frequency",
    "monetary",
    "recency_score",
    "frequency_score",
    "monetary_score",
    "RFM_SCORE",
    "Segment"
)

dim_customer.select(
    "cust_key",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "RFM_SCORE",
    "Segment"
).show(20, truncate=False)

+--------+--------------------------------+---------------------+--------------+---------+-------------------+
|cust_key|customer_unique_id              |customer_city        |customer_state|RFM_SCORE|Segment            |
+--------+--------------------------------+---------------------+--------------+---------+-------------------+
|1       |0000366f3b9a7992bf8c76cfdf3221e2|cajamar              |SP            |424      |Potential Loyalists|
|2       |0000b849f77a49e4a4ce2b2a4ca5be3f|osasco               |SP            |421      |Potential Loyalists|
|3       |0000f46a3911fa3c0805444483337064|sao jose             |SC            |152      |Others             |
|4       |0000f6ccb0745a6a4b88665a16c9f078|belem                |PA            |241      |At Risk            |
|5       |0004aac84e0df4da2b147fca70cf8255|sorocaba             |SP            |244      |At Risk            |
|6       |0004bd2a26a76fe21f786e4fbd80607f|sao paulo            |SP            |424      |Potential Loyalists|
|

In [101]:
dim_customer.write.mode("overwrite").parquet(
    "/ecom_modeling/output/dim_customer"
)

In [102]:
order_payments_cleaned.write \
    .mode("overwrite") \
    .parquet("/ecom_modeling/output/order_payments_cleaned")